In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [2]:
import os

print("CSV files in project:")
for file in os.listdir():
    if file.endswith(".csv"):
        print(file)

CSV files in project:
churn_predictions.csv
cleaned_statsfinal.csv
day14_performance_comparison.csv
feature_engineered_statsfinal.csv
model_comparison.csv
movie_recommendations.csv
prediction_outputs.csv
regression_coefficients.csv
regression_predictions.csv
statsfinal.csv
students.csv
WA_FnUseC_TelcoCustomerChurn.csv


In [3]:
movies = pd.DataFrame({
    "Movie": [
        "Inception",
        "Interstellar",
        "The Matrix",
        "Avatar",
        "Titanic",
        "The Dark Knight",
        "Avengers",
        "Jurassic Park"
    ],
    "Action": [9, 6, 9, 8, 3, 10, 9, 8],
    "SciFi": [10, 10, 10, 9, 2, 4, 8, 8],
    "Romance": [3, 2, 2, 4, 10, 2, 3, 2],
    "Adventure": [7, 8, 6, 10, 5, 7, 9, 10],
    "Popularity": [9, 9, 8, 9, 10, 10, 10, 9]
})

movies.to_csv("movie_recommendations.csv", index=False)

print("Recommendation dataset created successfully!")
print(movies)

Recommendation dataset created successfully!
             Movie  Action  SciFi  Romance  Adventure  Popularity
0        Inception       9     10        3          7           9
1     Interstellar       6     10        2          8           9
2       The Matrix       9     10        2          6           8
3           Avatar       8      9        4         10           9
4          Titanic       3      2       10          5          10
5  The Dark Knight      10      4        2          7          10
6         Avengers       9      8        3          9          10
7    Jurassic Park       8      8        2         10           9


In [4]:
df = pd.read_csv("movie_recommendations.csv")

print("Dataset loaded successfully!")
print(df)

Dataset loaded successfully!
             Movie  Action  SciFi  Romance  Adventure  Popularity
0        Inception       9     10        3          7           9
1     Interstellar       6     10        2          8           9
2       The Matrix       9     10        2          6           8
3           Avatar       8      9        4         10           9
4          Titanic       3      2       10          5          10
5  The Dark Knight      10      4        2          7          10
6         Avengers       9      8        3          9          10
7    Jurassic Park       8      8        2         10           9


In [5]:
features = ["Action", "SciFi", "Romance", "Adventure", "Popularity"]

X = df[features]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Features scaled successfully!")
print(X_scaled)

Features scaled successfully!
[[ 0.59339083  0.84051051 -0.19611614 -0.43759497 -0.37796447]
 [-0.83074716  0.84051051 -0.58834841  0.14586499 -0.37796447]
 [ 0.59339083  0.84051051 -0.58834841 -1.02105494 -1.88982237]
 [ 0.11867817  0.48661135  0.19611614  1.31278492 -0.37796447]
 [-2.25488515 -1.9906828   2.54950976 -1.60451491  1.13389342]
 [ 1.06810349 -1.28288447 -0.58834841 -0.43759497  1.13389342]
 [ 0.59339083  0.13271219 -0.19611614  0.72932496  1.13389342]
 [ 0.11867817  0.13271219 -0.58834841  1.31278492 -0.37796447]]


In [6]:
knn = NearestNeighbors(n_neighbors=4, metric="euclidean")

knn.fit(X_scaled)

print("KNN model trained successfully!")

KNN model trained successfully!


In [7]:
k_values = [2, 3, 4, 5, 6]

for k in k_values:
    knn_test = NearestNeighbors(n_neighbors=k, metric="euclidean")
    knn_test.fit(X_scaled)
    
    distances, indices = knn_test.kneighbors(X_scaled)
    
    avg_distance = distances[:, 1:].mean()
    
    print("K =", k, "| Average distance =", round(avg_distance, 3))

K = 2 | Average distance = 1.875
K = 3 | Average distance = 2.099
K = 4 | Average distance = 2.229
K = 5 | Average distance = 2.345
K = 6 | Average distance = 2.493


In [8]:
movie_name = "Inception"

movie_index = df[df["Movie"] == movie_name].index[0]

distances, indices = knn.kneighbors(
    X_scaled[movie_index].reshape(1, -1)
)

print("Recommendations similar to", movie_name)

for i in range(1, len(indices[0])):
    recommended_index = indices[0][i]
    distance = distances[0][i]
    
    print(
        df.iloc[recommended_index]["Movie"],
        "| Distance:",
        round(distance, 3)
    )

Recommendations similar to Inception
Interstellar | Distance: 1.588
The Matrix | Distance: 1.667
Avatar | Distance: 1.889


In [9]:
comparison = []

for k in k_values:
    knn_test = NearestNeighbors(n_neighbors=k)
    knn_test.fit(X_scaled)

    distances, indices = knn_test.kneighbors(X_scaled)

    avg_distance = distances[:, 1:].mean()

    comparison.append({
        "K": k,
        "Average Distance": round(avg_distance, 3)
    })

comparison_df = pd.DataFrame(comparison)

print(comparison_df)

comparison_df.to_csv("k_comparison.csv", index=False)

   K  Average Distance
0  2             1.875
1  3             2.099
2  4             2.229
3  5             2.345
4  6             2.493


## Best K Value

After comparing different K values, K = 4 was selected for the recommendation model.

A moderate K value provides a useful balance between finding sufficiently similar movies and avoiding recommendations that are too broad. The KNN model uses similarity between movie features such as Action, SciFi, Romance, Adventure, and Popularity.

In [11]:
best_k = 4

print("Best K value:", best_k)
print("Model selected based on recommendation similarity and balanced neighborhood size.")

Best K value: 4
Model selected based on recommendation similarity and balanced neighborhood size.
